In [ ]:
import pyomo.environ as pyomo
from pyomo.environ import SolverFactory, Suffix # ConcreteModel, Var, Objective, Constraint, NonNegativeReals, maximize
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
order_book_sheets = pd.read_excel('orderbook_2026.xlsx', sheet_name=None)
order_book = pd.concat(order_book_sheets.values(), ignore_index=True)
order_book

In [ ]:
def process_order_book(order_book):
    processed_book = {}
    #processed_book = pd.DataFrame(columns=["timestamp", "zone_prices", "zone_quantities"])
    
    for _, row in order_book.iterrows():
        order_id = row.order_id.split('_')
        is_supply = order_id[0] == 'S'
        originating_zone_id = 0 if order_id[1] == 'A' else 1
        timestamp = int(order_id[2])
        submit_order = int(order_id[3])
        
        if timestamp not in processed_book:
            processed_book[timestamp] = {"timestamp": timestamp, "supply_prices": [[],[]], "supply_quantities": [[],[]], "demand_prices": [[],[]], "demand_quantities": [[],[]]}

        if is_supply:
            processed_book[timestamp]['supply_prices'][originating_zone_id].append(row.price)
            processed_book[timestamp]['supply_quantities'][originating_zone_id].append(row.quantity)
        else:
            processed_book[timestamp]['demand_prices'][originating_zone_id].append(row.price)
            processed_book[timestamp]['demand_quantities'][originating_zone_id].append(row.quantity)
    
    return pd.DataFrame.from_dict(processed_book, orient='index')
        
order_book = process_order_book(order_book)
order_book

In [ ]:
print(len(order_book.iloc[0]['supply_prices'][0]))
print(len(order_book.iloc[0]['demand_prices'][0]))

## Q3 - Pyomo model implementation

In [ ]:
# create model object
model = pyomo.ConcreteModel()

# define sets
model.T = pyomo.Set(initialize=range(1, 97))
model.T0 = pyomo.Set(initialize=[1])
model.Tp = pyomo.Set(initialize=range(2, 97))
model.I = pyomo.Set(initialize=range(1, 21))
model.J = pyomo.Set(initialize=range(1, 16))
model.Z = pyomo.Set(initialize=[0,1])
model.Z0 = pyomo.Set(initialize=[0])

# define parameters
model.Sbase = pyomo.Param(initialize=100.0) # 100 MVA base
model.DeltaT = pyomo.Param(initialize=0.25) # quarter hour per timestep
model.Fmax = pyomo.Param(initialize=0.0, mutable=True) # interconnect capacity in MW
model.X = pyomo.Param(initialize=0.1) # reactance of interconnect in pu

def Q_offer_init(model, z, t, i):
    return order_book.loc[t, 'supply_quantities'][z][i-1]
model.Q_o = pyomo.Param(model.Z, model.T, model.I, initialize=Q_offer_init)

def pi_offer_init(model, z, t, i):
    return order_book.loc[t, 'supply_prices'][z][i-1]
model.pi_o = pyomo.Param(model.Z, model.T, model.I, initialize=pi_offer_init)

def Q_bid_init(model, z, t, i):
    return order_book.loc[t, 'demand_quantities'][z][i-1]
model.Q_b = pyomo.Param(model.Z, model.T, model.J, initialize=Q_bid_init)

def pi_bid_init(model, z, t, i):
    return order_book.loc[t, 'demand_prices'][z][i-1]
model.pi_b = pyomo.Param(model.Z, model.T, model.J, initialize=pi_bid_init)

model.K = pyomo.Param(model.Z, initialize=[0.0, 0.0], mutable=True) # ESS max power per zone in MW
model.SOE_min = pyomo.Param(model.Z, initialize=[10.0, 10.0]) # minimum SOE per zone in MWh
model.SOE_max = pyomo.Param(model.Z, initialize=[500.0, 500.0]) # maximum SOE per zone in MWh
model.SOE_ini = pyomo.Param(model.Z, initialize=[250.0, 250.0]) # initial SOE per zone in MWh
model.eta = pyomo.Param(model.Z, initialize=[0.95, 0.95]) # ESS efficiency per zone

In [ ]:
# define decision varaibles
# unrestricted real variables
model.delta = pyomo.Var(model.Z, model.T, within=pyomo.Reals) # voltage angle in rad
model.F = pyomo.Var(model.T, within=pyomo.Reals) # power flow over the connection in MW

# non-negative real variables
model.q_o = pyomo.Var(model.Z, model.T, model.I, within=pyomo.NonNegativeReals) # quantity of order accepted per zone and order in MWh
model.q_b = pyomo.Var(model.Z, model.T, model.J, within=pyomo.NonNegativeReals) # quantity of bid accepted per zone and bid in MWh
model.S = pyomo.Var(model.Z, model.T, within=pyomo.NonNegativeReals) # total quantity of accepted offers per zone in MWh
model.D = pyomo.Var(model.Z, model.T, within=pyomo.NonNegativeReals) # total quantity of accepted bids per zone in MWh
model.P_dis = pyomo.Var(model.Z, model.T, within=pyomo.NonNegativeReals) # ESS discharging power per zone in MW
model.P_ch = pyomo.Var(model.Z, model.T, within=pyomo.NonNegativeReals) # ESS charging power per zone in MW
model.SOE = pyomo.Var(model.Z, model.T, within=pyomo.NonNegativeReals) # SOE per zone in MWh

In [ ]:
# define model objective
def model_objective(model):
    value_of_bids = sum(model.pi_b[z,t,j] * model.q_b[z,t,j] for z in model.Z for t in model.T for j in model.J)
    cost_of_offers = sum(model.pi_o[z,t,i] * model.q_o[z,t,i] for z in model.Z for t in model.T for i in model.I)
    return value_of_bids - cost_of_offers

model.objective = pyomo.Objective(rule=model_objective, sense=pyomo.maximize)

In [ ]:
# supply and demand constraints
def rule_total_accepted_offers(model, z, t):
    return model.S[z,t] == sum(model.q_o[z,t,i] for i in model.I)
model.total_accepted_offers_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_total_accepted_offers)

def rule_max_offer(model, z, t, i):
    return model.q_o[z,t,i] <= model.Q_o[z,t,i]
model.max_offer_constraint = pyomo.Constraint(model.Z, model.T, model.I, rule=rule_max_offer)

def rule_total_accepted_bids(model, z, t):
    return model.D[z,t] == sum(model.q_b[z,t,j] for j in model.J)
model.total_accepted_bids_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_total_accepted_bids)

def rule_max_bid(model, z, t, j):
    return model.q_b[z,t,j] <= model.Q_b[z,t,j]
model.max_bid_constraint = pyomo.Constraint(model.Z, model.T, model.J, rule=rule_max_bid)

In [ ]:
# interconnection constraints
def rule_initial_angle(model, z, t):
    return model.delta[z,t] == 0.0
model.initial_angle_constraint = pyomo.Constraint(model.Z0, model.T, rule=rule_initial_angle)

def rule_max_flow(model, t):
    return model.F[t] <= model.Fmax
model.flow_max_constraint = pyomo.Constraint(model.T, rule=rule_max_flow)

def rule_min_flow(model, t):
    return model.F[t] >= -model.Fmax
model.flow_min_constraint = pyomo.Constraint(model.T, rule=rule_min_flow)

def rule_dc_power_flow(model, t):
    return model.F[t] == (((model.delta[0,t] - model.delta[1,t]) / model.X) * model.Sbase)
model.dc_power_flow_constraint = pyomo.Constraint(model.T, rule=rule_dc_power_flow)

In [ ]:
# VPL constraints
def rule_max_charging(model, z, t):
    return model.P_ch[z,t] <= model.K[z]
model.charging_max_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_max_charging)

def rule_max_discharging(model, z, t):
    return model.P_dis[z,t] <= model.K[z]
model.discharging_max_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_max_discharging)

def rule_max_soe(model, z, t):
    return model.SOE[z,t] <= model.SOE_max[z]
model.max_soe_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_max_soe)

def rule_min_soe(model, z, t):
    return model.SOE[z,t] >= model.SOE_min[z]
model.min_soe_constraint = pyomo.Constraint(model.Z, model.T, rule=rule_min_soe)

def rule_soe_level(model, z, t):
    return model.SOE[z,t] == model.SOE[z,t-1] + (model.eta[z] * model.P_ch[z,t] * model.DeltaT - (model.P_dis[z,t] / model.eta[z]) * model.DeltaT)
model.soe_level_constraint = pyomo.Constraint(model.Z, model.Tp, rule=rule_soe_level)

def rule_soe_level_ini(model, z, t):
    return model.SOE[z,t] == model.SOE_ini[z] + (model.eta[z] * model.P_ch[z,t] * model.DeltaT - (model.P_dis[z,t] / model.eta[z]) * model.DeltaT)
model.soe_level_ini_constraint = pyomo.Constraint(model.Z, model.T0, rule=rule_soe_level_ini)

def rule_vpl_link1(model, t):
    return model.P_ch[0, t] == model.P_dis[1, t]
model.vpl_link1_constraint = pyomo.Constraint(model.T, rule=rule_vpl_link1)

def rule_vpl_link2(model, t):
    return model.P_ch[1, t] == model.P_dis[0, t]
model.vpl_link2_constraint = pyomo.Constraint(model.T, rule=rule_vpl_link2)

In [ ]:
# zonal energy balance
def rule_zonal_balance_A(model, t):
    return model.S[0,t] + model.P_dis[0,t] * model.DeltaT == model.D[0,t] + model.P_ch[0,t] * model.DeltaT + model.F[t] * model.DeltaT
model.zonal_balance_A_constraint = pyomo.Constraint(model.T, rule=rule_zonal_balance_A)

def rule_zonal_balance_B(model, t):
    return model.S[1,t] + model.P_dis[1,t] * model.DeltaT + model.F[t] * model.DeltaT == model.D[1,t] + model.P_ch[1,t] * model.DeltaT
model.zonal_balance_B_constraint = pyomo.Constraint(model.T, rule=rule_zonal_balance_B)

In [ ]:
# scenarios
scenarios = {
    1: {'Fmax': 0, 'K': 0},
    2: {'Fmax': 1000, 'K': 0},
    3: {'Fmax': 180, 'K': 0},
    4: {'Fmax': 180, 'K': 50}
}

In [ ]:
# solving model
solver= SolverFactory('gurobi')
solver_results = {}
for i, params in scenarios.items():
    model.Fmax = params['Fmax']
    model.K[0] = params['K']
    model.K[1] = params['K']

    solver.solve(model)
    solver_results[i] = [pyomo.value(model.objective()) for t in model.T] 

In [ ]:
# display optimal objective function value for each scenario
for i, params in scenarios.items():
    print(f'Optimal objective function value for scenario {i}: {solver_results[i][0]:.2f}')

## Q4

In [ ]:
# interconnector power flow over time in Scenarios 1-4

# solve model 
solver = pyomo.SolverFactory('gurobi')
flow_results = {}

for i, params in scenarios.items():
    model.Fmax = params['Fmax']
    model.K[0] = params['K']
    model.K[1] = params['K']

    solver.solve(model)
    flow_results[i] = [pyomo.value(model.F[t]) for t in model.T] 

# plot interconnector power flow over time for each scenario
plt.figure(figsize=(10,6))
for i, params in scenarios.items():
    plt.plot(flow_results[i], label=f'Scenario {i}')
plt.xlabel('Time')
plt.ylabel('Power flow')
plt.title('Interconnector power flow over time')
plt.legend()
plt.show()

# identify periods when the interconnector is congested for scenarios 3 and 4
for i in [3,4]:
    periods = []
    interconnector_congested = [abs(f) >= scenarios[i]['Fmax'] - 1e-6 for f in flow_results[i]]
    start = None

    for j in range(len(interconnector_congested)):
        if interconnector_congested[j] and start is None:
            start = list(model.T)[j]

        elif not interconnector_congested[j] and start is not None:
            periods.append((start,list(model.T)[j-1]))
            start = None

    if start is not None:
        periods.append((start,list(model.T)[-1])) 

    print(f'Scenario {i}: periods:{periods}')


## Q5

In [ ]:
# electricity price and supply and demand volumes in each area over time in Scenarios 1-4

# solve model 
solver = pyomo.SolverFactory('gurobi')
electricity_results = {}
supply_results = {}
demand_results = {}
    # define suffix to import dual values
model.dual = Suffix(direction=Suffix.IMPORT)

for i, params in scenarios.items():
    model.Fmax = params['Fmax']
    model.K[0] = params['K']
    model.K[1] = params['K']

    solver.solve(model)
    electricity_results[i] = { 0: {t: -model.dual[model.zonal_balance_A_constraint[t]] for t in model.T}, 1: {t: -model.dual[model.zonal_balance_B_constraint[t]] for t in model.T} }
    supply_results[i] = { z: {t: pyomo.value(model.S[z,t]) for t in model.T} for z in model.Z }
    demand_results[i] = { z: {t: pyomo.value(model.D[z,t]) for t in model.T} for z in model.Z}

# plot electricity price in each area over time for each scenario
#define mapping of zone index to name
zone_name = {
    0: "A",
    1: "B"
}

plt.figure(figsize=(10,6))
for i, params in scenarios.items():
    for z in model.Z:
        plt.plot([electricity_results[i][z][t] for t in model.T], label=f"Zone {zone_name[z]}, Scenario {i}")
plt.xlabel('Time')
plt.ylabel('Electricity price')
plt.title('Electricity prices in each area over time')
plt.legend()
plt.show()

# electricity price calculations
for i, params in scenarios.items():
        for z in model.Z:
            print(f'Minimum electricity price in zone {zone_name[z]} for scenario {i}: {min([electricity_results[i][z][t] for t in model.T])}')
            print(f'Average electricity price in zone {zone_name[z]} for scenario {i}: {np.mean([electricity_results[i][z][t] for t in model.T])}')
            print(f'Maximum electricity price in zone {zone_name[z]} for scenario {i}: {max([electricity_results[i][z][t] for t in model.T])}')
            print("\n")


# plot difference between supply and demand volumes over time for each scenario
plt.figure(figsize=(10,6))
for i, params in scenarios.items():
    for z in model.Z:
        difference = [supply_results[i][z][t] - demand_results[i][z][t] for t in model.T]
        plt.plot(difference, label=f"Zone {zone_name[z]}, Scenario {i}")
plt.xlabel('Time')
plt.ylabel('Volume difference')
plt.title('Difference between supply and demand volumes over time')
plt.legend()
plt.show()


## Q6

In [ ]:
# state-of-enrgy flow and net power of the two ESS over time for scenario 4

# solve model 
solver = pyomo.SolverFactory('gurobi')
state_of_energy_results = {}
discharging_results = {}
charging_results = {}

model.Fmax = 180
model.K[0] = 50
model.K[1] = 50

solver.solve(model) 
state_of_energy_results = { z: {t: pyomo.value(model.SOE[z,t]) for t in model.T} for z in model.Z }
discharging_results = { z: {t: pyomo.value(model.P_dis[z,t]) for t in model.T} for z in model.Z } 
charging_results= { z: {t: pyomo.value(model.P_ch[z,t]) for t in model.T} for z in model.Z }


# plot state-of-energy 
#define mapping of zone index to name
ESS_name = {
    0: "ESS-1",
    1: "ESS-2"
}

# boolean to determine congestion of the interconnector
net = { z: [charging_results[z][t] - discharging_results[z][t] for t in model.T ] for z in model.Z }
congestion = [x > 1e-6 for x in net[0]]

plt.figure(figsize=(10,6))
for z in model.Z:
    plt.plot(list(model.T),[state_of_energy_results[z][t] for t in model.T], label=f"{ESS_name[z]}")
for i, t in enumerate(model.T):
    if congestion[i] > 1e-6:
        plt.axvspan(t-1 , t  , color='red', alpha=0.15)
plt.xlabel('Time')
plt.ylabel('State of energy')
plt.title('State of energy over time in scenario 4')
plt.plot([], []  , color='red', alpha=0.15, label = 'Interconnector congestion')
plt.legend()
plt.show()

# plot net power

plt.figure(figsize=(10,6))
for z in model.Z:
    plt.plot(list(model.T),net[z], label=f"{ESS_name[z]}")
plt.xlabel('Time')
plt.ylabel('Net power')
plt.title('Net power over time in scenario 4')
plt.legend()
plt.show()

## Q7

In [ ]:
#increasing rated power of the VPL
# define new scenarios
new_scenarios = {
    1: {'Fmax': 180, 'K': 50},
    2: {'Fmax': 180, 'K': 100},
    3: {'Fmax': 180, 'K': 150},
    4: {'Fmax': 180, 'K': 200},
    5: {'Fmax': 180, 'K': 250},
    6: {'Fmax': 180, 'K': 300},
    7: {'Fmax': 180, 'K': 350},
    8: {'Fmax': 500, 'K': 0}        # increasing capacity of the interconnection and retirering the VPL
}


# solve model 
solver = pyomo.SolverFactory('gurobi')
new_electricity_results = {}
new_flow_results = {}
new_state_of_energy_results = {}

for i, params in new_scenarios.items():
    model.Fmax = params['Fmax']
    model.K[0] = params['K']
    model.K[1] = params['K']

    solver.solve(model)
    new_electricity_results[i] = { 0: {t: -model.dual[model.zonal_balance_A_constraint[t]] for t in model.T}, 1: {t: -model.dual[model.zonal_balance_B_constraint[t]] for t in model.T} }
    new_flow_results[i] = [pyomo.value(model.F[t]) for t in model.T]
    new_state_of_energy_results [i] = { z: {t: pyomo.value(model.SOE[z,t]) for t in model.T} for z in model.Z }

# plot electricity price in each area over time for each scenario
plt.figure(figsize=(10,6))
for i, params in new_scenarios.items():
    for z in model.Z:
        plt.plot([new_electricity_results[i][z][t] for t in model.T], label=f"Zone {zone_name[z]}, Scenario {i}")
plt.xlabel('Time')
plt.ylabel('Electricity price')
plt.title('Electricity prices in each area over time')
plt.legend()
plt.show()

# average price spread for each K
for i in new_scenarios:
    spread = np.mean([new_electricity_results[i][1][t]-new_electricity_results[i][0][t] for t in model.T])
    print(f'K = {new_scenarios[i]['K']}: average price spread = {spread:.2f}')

# interconnector power flow
plt.figure(figsize=(10,6))
for i, params in new_scenarios.items():
    plt.plot(new_flow_results[i], label=f'Scenario {i}')
plt.xlabel('Time')
plt.ylabel('Power flow')
plt.title('Interconnector power flow over time')
plt.legend()
plt.show()

# state-of-energy flow 
plt.figure(figsize=(10,6))
for i, params in new_scenarios.items():
    for z in model.Z:
        plt.plot(list(model.T),[new_state_of_energy_results[i][z][t] for t in model.T], label=f"{ESS_name[z]}, Scenario {i}")
plt.xlabel('Time')
plt.ylabel('State of energy')
plt.title('State of energy over time ')
plt.legend()
plt.show()

## Q8